# Previsão de volume do Sistema Cantareira (CT_TOTAL) via XGBoost — artigo (H = 3)

Notebook do braço XGBoost do artigo: gera a *ablation* dos feature sets
A/B/C/D nas duas bases de reconstrução, a tabela de desempenho contra
persistência e climatologia, o teste de Diebold–Mariano, as figuras de séries
e a interpretabilidade via **SHAP** (`TreeExplainer`). A correlação das
candidatas (§3.3) e o SHAP (§5) também orientaram a composição dos feature sets
C e D da LSTM ([`lstm_article.ipynb`](../LSTM/lstm_article.ipynb) §4).

**Protocolo.** Split fixo treino (até dez/2015) / validação (jan/2016–out/2018)
/ teste (nov/2018–dez/2025, 86 meses), com `split_on="target_date"`. O
walk-forward expansivo é o protocolo do braço LSTM; aqui o XGBoost é ajustado
uma vez.

**Fora deste notebook.** As análises por reservatório, a *ablation* `level` × `delta`, o PDP/ICE e a análise de
tendência do TWS estão em [`xgboost_exploratory.ipynb`](xgboost_exploratory.ipynb);
nada aqui depende delas.

**Saídas** em `Previsao/Outputs/XGBoost/` com o prefixo `article_ct_total_h3_*`.


## § 1. Ambiente e configuração

Reusa `Previsao.Scripts` (mesmos módulos do braço XGBoost usado em toda a
pesquisa) — nenhum código de modelagem novo.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd().resolve()
# Sobe até a raiz do repositório (a pasta que contém 'Previsao/'),
# robusto a qualquer cwd e ao nome do diretório do clone.
REPO_ROOT = next(
    (p for p in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents)
     if (p / 'Previsao').is_dir()),
    NOTEBOOK_DIR,
)
PREVISAO_DIR = REPO_ROOT / 'Previsao'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Previsao.Scripts import (
    XGBoostModel,
    FeatureConstructor,
    default_cantareira_total_spec, build_total_cantareira_inputs,
    get_experiments_abc, run_experiments,
    seasonal_climatology_baseline,
    fit_single_experiment_for_shap, compute_tree_shap,
    shap_importance_table, shap_group_importance_table,
    save_shap_beeswarm,
    set_paper_plot_style, save_figure,
    plot_shap_importance_bar, plot_shap_group_bar,
    diebold_mariano_test,
)

DATA_DIR = PREVISAO_DIR / "Dados"
OUT_DIR = PREVISAO_DIR / "Outputs" / "XGBoost"
FIG_DIR = OUT_DIR / "figures"
PRED_DIR = OUT_DIR / "predictions"
SHAP_DIR = OUT_DIR / "shap_importance"

for p in [FIG_DIR, FIG_DIR / "shap", PRED_DIR, SHAP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

set_paper_plot_style()
print("Ambiente OK.")
print(f"  REPO_ROOT = {REPO_ROOT}")
print(f"  OUT_DIR   = {OUT_DIR}")

In [ ]:
CONFIG = {
    "H": 3,
    "MIN_DATE": "1984-01-01",
    "MAX_DATE": "2025-12-01",
    "SPLIT_MODE": "3way",
    "SPLIT_ON": "target_date",
    "TRAIN_END": "2015-12-01",
    "VAL_END": "2018-10-01",
    "LAGS_FEATURES": [1, 2, 3, 6, 12],
    "ROLL_WINDOWS": [2, 3, 6, 12],
    "INDEX_SCALES": [3, 6, 12],
    "CORE_REQUIRED_MIN": [
        "pr", "eto", "q", "Vol", "Vol_Lag1",
        "month_sin", "month_cos", "is_dry",
    ],
    "SCALE_RAW_VARS": [
        "pr", "eto", "q", "Vol", "qout", "qbal", "tws", "qpbs_net", "qelev",
    ],
    "TARGET_LAGS": [1, 3, 6, 12],
}
print(pd.Series(CONFIG, dtype=object).to_string())

## § 2. Dados e agregação CT_TOTAL

CSVs mensais de `Previsao/Dados/`. Os 4 reservatórios são agregados em
CT_TOTAL: soma para vazões e volume; média ponderada por área
(`CANTAREIRA_AREA_KM2`) para Pr/ETo/TWS.

In [ ]:
def std(rel_path: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / rel_path)
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)


pr = std("Pr/Pr_monthly.csv")
eto = std("ETo/ETo_monthly.csv")
qnat = std("Q/Qnat2_monthly.csv")
vol = std("Vol/Vol_monthly.csv")
qbal = std("Q/Qbal_monthly.csv")
qout = std("Q/Qout_monthly.csv")
qt5 = std("Q/Qt5_monthly.csv")
qt6 = std("Q/Qt6_monthly.csv")
qt7 = std("Q/Qt7_monthly.csv")
qpbs = std("Q/Qpbs_net_monthly.csv")
qelev = std("Q/Qelev_monthly.csv")
tws = std("TWS/TWS_monthly.csv")

eq_total = build_total_cantareira_inputs(
    pr=pr, eto=eto, qnat=qnat, vol=vol,
    qbal=qbal, qout=qout, tws=tws,
    spec=default_cantareira_total_spec(),
)
vol_total = eq_total["vol"]
print(f"CT_TOTAL: {vol_total['date'].min().date()} a "
      f"{vol_total['date'].max().date()}, n={len(vol_total)}")

### 2.1. Engenharia de features

`FeatureConstructor` com lags {1,2,3,6,12}, médias móveis {2,3,6,12} e
índices padronizados (SPI/SPEI/SRI) em {3,6,12} meses.

In [ ]:
fc_total = FeatureConstructor(
    basins=["CT_TOTAL"],
    lags=CONFIG["LAGS_FEATURES"],
    roll_windows=CONFIG["ROLL_WINDOWS"],
    index_scales=CONFIG["INDEX_SCALES"],
    stats_ops=["Mean"],
)
_, datasets_total = fc_total.build(
    pr=eq_total["pr"], eto=eq_total["eto"], qnat=eq_total["qnat"],
    qbal=eq_total["qbal"], qout=eq_total["qout"], tws=eq_total["tws"],
    qt5=qt5, qt6=qt6, qt7=qt7, qpbs_net=qpbs, qelev=qelev,
)
print(f"CT_TOTAL dataset: {datasets_total['CT_TOTAL'].shape}")

## § 3. Experimentos A/B/C/D × duas bases de reconstrução

**A** e **B** vêm de `get_experiments_abc` (mesma função usada pelo XGBoost
em toda a pesquisa); **C** e **D** são definidos localmente (iguais aos da
LSTM):

* **A_core**: hidroclimáticas cruas + volume + lags imediatos + calendário.
* **B_external_ops**: A + variáveis operacionais externas (`qpbs_net`,
  `qelev`; túneis internos se cancelam no agregado).
* **C — A + `qbal`**: núcleo A + o balanço operacional agregado.
* **D — A + `qbal` + `SRI12`**: C + o índice padronizado de vazão de 12 meses.

**Duas bases de reconstrução — ambas preveem uma VARIAÇÃO.** O modo `level`
cru (prever o volume direto) foi **removido** desta análise:

* **persistência** (A/B/C/D): alvo Δ = Vol(t+H) − Vol(t); reconstrução
  V̂ = Vol(t) + Δ̂.
* **climatologia** (A\*/B\*/C\*/D\*): alvo = anomalia sazonal
  Vol(t+H) − Clim[m(t+H)] (nível sobre a série de anomalias); reconstrução
  V̂ = Clim[m(t+H)] + Δ̂. Como no LSTM, a feature `Vol` passa a ser a anomalia
  do mês corrente. A climatologia usa só o treino (sem vazamento).

Escolhe-se a combinação de **menor RMSE de teste em espaço físico** entre os
8 experimentos.

In [ ]:
EXP_SHORT = {"A_core": "A", "B_external_ops": "B", "C": "C", "D": "D"}


def feature_sets_ct_total() -> dict[str, list[str]]:
    sets = get_experiments_abc(basin="CT_TOTAL")
    a = sets["A_core_raw"]
    # C = A + qbal; D = A + qbal + SRI12 (mesmos feature sets do LSTM).
    return {
        "A_core": a,
        "B_external_ops": sets["B_core_raw_plus_ops"],
        "C": a + ["qbal"],
        "D": a + ["qbal", "SRI12"],
    }


# Climatologia mensal do volume no TREINO (date <= TRAIN_END): base sazonal
# dos experimentos "*". Sem vazamento (só treino).
_train_m = vol_total["date"] <= pd.Timestamp(CONFIG["TRAIN_END"])
CLIM_M = (vol_total.loc[_train_m]
          .groupby(vol_total["date"].dt.month)["CT_TOTAL"].mean())

# Série de anomalias VolAnom(t) = Vol(t) - Clim[m(t)] (base climatologia).
vol_anom = vol_total.copy()
vol_anom["CT_TOTAL"] = (vol_total["CT_TOTAL"]
                        - vol_total["date"].dt.month.map(CLIM_M))


def run_ct_total_base(base: str) -> pd.DataFrame:
    """Roda os 4 feature sets numa das duas bases de reconstrução (delta).

    base='pers'  -> alvo delta sobre Vol; reconstrução Vol(t) + Δ̂ (A/B/C/D).
    base='clim'  -> alvo sobre a anomalia sazonal (level em VolAnom);
                    reconstrução Clim[m(t+H)] + Δ̂ (A*/B*/C*/D*).
    Ambas preveem uma VARIAÇÃO; o modo 'level' cru (prever Vol direto) foi
    removido desta análise.
    """
    tmode = "delta" if base == "pers" else "level"
    vols = vol_total if base == "pers" else vol_anom
    xgb = XGBoostModel(target_col="Vol", add_seasonality=True,
                       target_mode=tmode)
    _, preds = run_experiments(
        xgb=xgb, datasets_by_basin=datasets_total, volumes=vols,
        basin="CT_TOTAL", horizon=CONFIG["H"],
        min_date=CONFIG["MIN_DATE"], max_date=CONFIG["MAX_DATE"],
        split_mode=CONFIG["SPLIT_MODE"], train_end=CONFIG["TRAIN_END"],
        val_end=CONFIG["VAL_END"], feature_sets=feature_sets_ct_total(),
        core_required=CONFIG["CORE_REQUIRED_MIN"],
        target_lags=CONFIG["TARGET_LAGS"], split_on=CONFIG["SPLIT_ON"],
        scale_mode="zscore", scale_raw_vars=CONFIG["SCALE_RAW_VARS"],
        scale_include_lags=True,
    )
    preds = preds.copy()
    preds["base"] = base
    preds["target_date"] = pd.to_datetime(preds["target_date"])
    if base == "clim":
        # Reconstrói o espaço físico: soma Clim[m(t+H)] às séries de anomalia.
        ct = preds["target_date"].dt.month.map(CLIM_M).to_numpy()
        for col in ("y_true", "y_pred", "y_base"):
            preds[col] = preds[col].to_numpy() + ct
    suffix = "" if base == "pers" else "*"
    preds["label"] = preds["experiment"].map(EXP_SHORT) + suffix
    return preds


preds_pers = run_ct_total_base("pers")
preds_clim = run_ct_total_base("clim")
preds_all = pd.concat([preds_pers, preds_clim], ignore_index=True)


def _rmse(a, b) -> float:
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


# Ablation em ESPAÇO FÍSICO (uniforme às duas bases), a partir das predições.
_rows = []
for (base, exp), g in preds_all.groupby(["base", "experiment"], sort=False):
    te = g[g["split"] == "test"]
    va = g[g["split"] == "val"]
    _rows.append({
        "label": g["label"].iloc[0], "base": base, "experiment": exp,
        "test_rmse": _rmse(te["y_true"], te["y_pred"]),
        "test_mae": float(np.mean(np.abs(te["y_true"].to_numpy()
                                         - te["y_pred"].to_numpy()))),
        "test_corr": float(te[["y_true", "y_pred"]].corr().iloc[0, 1]),
        "val_rmse": _rmse(va["y_true"], va["y_pred"]),
    })
ablation_ct_total = (pd.DataFrame(_rows)
                     .sort_values("test_rmse").reset_index(drop=True))
ablation_ct_total[["label", "base", "experiment",
                   "test_rmse", "test_mae", "test_corr", "val_rmse"]].round(3)

In [ ]:
best_row = ablation_ct_total.iloc[0]
BEST_BASE = best_row["base"]
BEST_EXP = best_row["experiment"]
BEST_LABEL = best_row["label"]
print(f"CT_TOTAL (H=3): melhor = {BEST_LABEL} (base={BEST_BASE}, {BEST_EXP}) "
      f"— RMSE teste = {best_row['test_rmse']:.2f}, "
      f"corr = {best_row['test_corr']:.3f}.")

preds_best = preds_all[(preds_all["base"] == BEST_BASE)
                       & (preds_all["experiment"] == BEST_EXP)].copy()
test_best = preds_best[preds_best["split"] == "test"].sort_values(
    "target_date").reset_index(drop=True)

# Baselines de teste (persistência-3 = Vol(t) e climatologia sazonal),
# tomados da base 'pers' (experimento A) e alinhados por target_date.
_ref = (preds_pers[(preds_pers["split"] == "test")
                   & (preds_pers["experiment"] == "A_core")]
        .sort_values("target_date").reset_index(drop=True))
persist3 = _ref["y_base"].to_numpy()                       # Vol(t)
clim_saz = _ref["target_date"].dt.month.map(CLIM_M).to_numpy()
assert np.allclose(_ref["y_true"].to_numpy(),
                   test_best["y_true"].to_numpy(), atol=1e-6), \
    "target_dates/observados divergem entre bases"


def _metrics_phys(y_true, y_pred) -> dict:
    o = np.asarray(y_true, float)
    p = np.asarray(y_pred, float)
    r = float(np.corrcoef(o, p)[0, 1])
    nse = 1.0 - float(np.sum((o - p) ** 2) / np.sum((o - o.mean()) ** 2))
    alpha = float(p.std() / o.std())
    beta = float(p.mean() / o.mean())
    kge = 1.0 - float(np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))
    return {"RMSE": _rmse(o, p), "MAE": float(np.mean(np.abs(o - p))),
            "r": r, "NSE": nse, "KGE": kge}


# Tabela de performance — os 8 experimentos + benchmarks (teste, espaço
# físico), ordenados por RMSE. r=Pearson, NSE=Nash-Sutcliffe, KGE=Kling-Gupta.
_yobs = _ref["y_true"].to_numpy()
_mean_train = float(vol_total.loc[
    vol_total["date"] <= pd.Timestamp(CONFIG["TRAIN_END"]), "CT_TOTAL"].mean())
_prows = []
for lab in ablation_ct_total["label"]:      # 8 experimentos, já em ordem de RMSE
    g = (preds_all[(preds_all["label"] == lab)
                   & (preds_all["split"] == "test")]
         .sort_values("target_date"))
    _prows.append({"preditor": lab, "tipo": "modelo",
                   **_metrics_phys(_yobs, g["y_pred"].to_numpy())})
for name, yhat in [("persistência-3", persist3),
                   ("climatologia sazonal", clim_saz),
                   ("média treino", np.full_like(_yobs, _mean_train))]:
    _prows.append({"preditor": name, "tipo": "benchmark",
                   **_metrics_phys(_yobs, yhat)})
perf_ct_total = pd.DataFrame(_prows).round(3)

# Comparação enxuta (vencedor vs baselines) mantida para referência rápida.
comparacao = perf_ct_total[
    perf_ct_total["preditor"].isin(
        [f"XGBoost ({BEST_LABEL})", BEST_LABEL,
         "persistência-3", "climatologia sazonal"])
][["preditor", "RMSE", "r"]].reset_index(drop=True)

perf_ct_total

### 3.1. Leitura da *ablation*

Com o modo `level` cru removido, a comparação é entre as **duas bases de
reconstrução** (ambas preveem uma variação). A **base persistência domina**
com folga em H=3: **`D` (A+qbal+SRI12) vence** (RMSE teste ≈ **61,2 hm³**,
corr ≈ **0,93**), seguido de `C` (64,9), `B` (65,5) e `A` (74,4) — todos **à
frente de qualquer** experimento da base climatologia (`A*` 83,7; `C*` 85,7;
`B*` 87,1; `D*` 93,5). Num sistema fortemente autocorrelacionado, no horizonte
de 3 meses o volume corrente `Vol(t)` é uma âncora melhor que a climatologia
sazonal. O vencedor bate com folga os baselines: persistência‑3 (≈ 126) e
climatologia sazonal (≈ 143). A **tabela de performance completa** (os 8
experimentos + benchmarks, com RMSE/MAE/r/NSE/KGE) é exibida acima.

*Nota:* o `SRI12` de `D` **ajuda na base persistência** (D melhor que C), mas
na base climatologia o `D*` é o **pior** dos quatro — a interação entre base
de reconstrução e feature set importa, e é por isso que ambas as bases são
mostradas (§4).

### 3.2. Significância da melhora — teste de Diebold–Mariano

O teste compara, mês a mês, o erro do vencedor com o de cada referência na série
de teste (86 meses), sob perda quadrática. Como as previsões de H = 3 se
sobrepõem, a variância do diferencial de perda inclui autocovariâncias até a
defasagem H − 1; a estatística recebe a correção de amostra pequena de Harvey et
al. (1997) e é comparada com uma t de Student com n − 1 graus de liberdade.
Diferencial médio negativo indica que o modelo erra menos que a referência.

Implementação em `Previsao.Scripts.forecast_evaluation.diebold_mariano_test`, a
mesma usada no `lstm_article.ipynb` (§9.2). O artigo reporta a comparação com a
persistência.


In [ ]:
# Teste DM: vencedor contra persistência-3 e climatologia sazonal, na série de
# teste em ordem temporal (alinhamento verificado no assert da célula anterior).
dm_ct_total = pd.DataFrame([
    {"modelo": BEST_LABEL, "referência": nome,
     **diebold_mariano_test(test_best["y_true"].to_numpy(),
                            test_best["y_pred"].to_numpy(), ref,
                            horizon=CONFIG["H"])}
    for nome, ref in [("persistência-3", persist3),
                      ("climatologia sazonal", clim_saz)]
])
dm_ct_total.round(4)

### 3.3. Seleção de C/D: correlação de todas as variáveis candidatas

O *heatmap* abaixo é a ferramenta de **seleção** dos experimentos C e D:
mostra a correlação de Pearson entre o alvo `Vol` e **todas as variáveis
candidatas** — as 8 variáveis-base (`pr`, `eto`, `q`, `qbal`, `qout`,
`qpbs_net`, `qelev`, `tws`) e os 9 índices padronizados (SPI/SPEI/SRI a
3/6/12 meses) — no CT_TOTAL e na janela do estudo, ordenadas por |ρ| com o
volume. Lags e médias móveis (derivadas mecânicas das bases) ficam de fora
por legibilidade — a memória multi-mês já entra pelos índices padronizados.
Os candidatos escolhidos — `qbal` (C) e `SRI12` (D) — aparecem destacados.

A leitura orienta a seleção e é **não** trivial:

* Entre os índices de vazão, o `SRI12` é o que mais correlaciona com o nível
  (ρ ≈ 0,75) — daí sua escolha como candidato de D.
* `qbal` é **quase ortogonal ao nível** (ρ ≈ 0,03), mas fortemente ligado à
  vazão (ρ ≈ 0,84 com `q`) — carrega informação de **fluxo**, não de estoque;
  por isso o SHAP (§5) o coloca como 3º driver apesar da correlação linear
  baixa.

A correlação linear ajuda a **mapear** o espaço de candidatos, mas não prediz
a importância no modelo (alvo `delta`) — a decisão final vem da *ablation*
(§3.1) e do SHAP (§5). Leitura interpretativa, não causal (`docs/methodological_rules.md` §3.6).

In [ ]:
import seaborn as sns

# Heatmap de correlação usado na SELEÇÃO de C/D: inclui TODAS as variáveis
# candidatas — alvo Vol + variáveis-base contemporâneas + todos os índices
# padronizados (SPI/SPEI/SRI 3/6/12). Lags e médias móveis são derivadas
# mecânicas das bases (altamente autocorrelacionadas) e ficam de fora por
# legibilidade; a memória multi-mês já entra pelos índices padronizados.
BASE_VARS = ["pr", "eto", "q", "qbal", "qout", "qpbs_net", "qelev", "tws"]
INDEX_VARS = ["SPI3", "SPI6", "SPI12", "SPEI3", "SPEI6", "SPEI12",
              "SRI3", "SRI6", "SRI12"]

_base = datasets_total["CT_TOTAL"].merge(
    vol_total.rename(columns={"CT_TOTAL": "Vol"})[["date", "Vol"]],
    on="date", how="left",
)
_base = _base[(_base["date"] >= CONFIG["MIN_DATE"])
              & (_base["date"] <= CONFIG["MAX_DATE"])]
pool = ["Vol"] + [v for v in BASE_VARS + INDEX_VARS if v in _base.columns]
corr_C = _base[pool].corr()
# Ordena: Vol primeiro, depois por |correlação com Vol| decrescente.
_ord = ["Vol"] + (corr_C["Vol"].drop("Vol").abs()
                   .sort_values(ascending=False).index.tolist())
corr_C = corr_C.loc[_ord, _ord]

# Variáveis que definem C (qbal) e D (qbal + SRI12), destacadas.
CAND_C = ["qbal", "SRI12"]

fig, ax = plt.subplots(figsize=(11.5, 10.0))
sns.heatmap(corr_C, ax=ax, cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            annot=True, fmt=".2f", annot_kws={"size": 6}, square=True,
            linewidths=.4, linecolor="white",
            cbar_kws={"label": "Pearson correlation", "shrink": .8})
for lab in (*ax.get_xticklabels(), *ax.get_yticklabels()):
    if lab.get_text() in CAND_C:
        lab.set_color("#ad5528"); lab.set_fontweight("bold")
ax.set_title("CT_TOTAL — correlation of all candidate variables "
             "(base + standardized indices); qbal + SRI12 (C/D) highlighted",
             loc="left", fontweight="bold")
fig.tight_layout()
save_figure(fig, FIG_DIR / "article_ct_total_h3_feature_correlation_C.png",
            dpi=300)
corr_C.to_csv(SHAP_DIR / "article_ct_total_h3_feature_correlation_C.csv")

print("Correlação com o alvo Vol (todas as candidatas, |ρ| decrescente):")
print(corr_C["Vol"].drop("Vol").round(3).to_string())
plt.show()

### 3.3.1. Material suplementar — correlação das features candidatas (Fig. S1)

A **Fig. S1** do material suplementar mostra a correlação de Pearson entre as
features candidatas descritas no manuscrito (Tabela 1, mais `qout` do conjunto B):
para pr, eto, q e as vazões operacionais, o valor do mês, lags 1/3/6/12 e médias
móveis 2/3/6/12; os índices SPI/SPEI/SRI a 3/6/12 meses; `tws` só no valor do
mês; e o calendário. Inclui o volume corrente `Vol`. As features ficam
agrupadas pela variável de entrada de que derivam (SPI→pr, SPEI→pr−eto, SRI→q).
Série mensal do CT_TOTAL, janela 1984-01→2025-12, correlação por pares completos
(o `tws` só existe de 2002 a 2024).

In [ ]:
# Fig. S1 (suplementar): correlação de Pearson entre as features candidatas
# do manuscrito (Tabela 1 + qout), agrupadas pela variável de entrada.
LAGS_S1 = [1, 3, 6, 12]
ROLLS_S1 = [2, 3, 6, 12]


def _derived(v: str) -> list[str]:
    """Valor do mês + lags + médias móveis da variável v (Tabela 1)."""
    return ([v] + [f"{v}_Lag{k}" for k in LAGS_S1]
            + [f"{v}_RollingMean{w}" for w in ROLLS_S1])


GROUPS_S1 = [  # (rótulo do grupo, features)
    ("Vol", ["Vol"]),
    ("pr", _derived("pr") + ["SPI3", "SPI6", "SPI12"]),
    ("eto", _derived("eto")),
    ("pr − eto", ["SPEI3", "SPEI6", "SPEI12"]),
    ("q", _derived("q") + ["SRI3", "SRI6", "SRI12"]),
    ("qout", _derived("qout")),
    ("qbal", _derived("qbal")),
    ("qpbs_net", _derived("qpbs_net")),
    ("qelev", _derived("qelev")),
    ("tws", ["tws"]),
    ("calendar", ["month_sin", "month_cos", "is_dry"]),
]

_s1 = datasets_total["CT_TOTAL"].merge(
    vol_total.rename(columns={"CT_TOTAL": "Vol"})[["date", "Vol"]],
    on="date", how="left",
)
_s1 = _s1[(_s1["date"] >= CONFIG["MIN_DATE"])
          & (_s1["date"] <= CONFIG["MAX_DATE"])].copy()
# Calendário com a mesma definição de XGBoostModel (add_seasonality).
_m = _s1["date"].dt.month
_s1["month_sin"] = np.sin(2.0 * np.pi * _m / 12.0)
_s1["month_cos"] = np.cos(2.0 * np.pi * _m / 12.0)
_s1["is_dry"] = ((_m >= 4) & (_m <= 9)).astype(int)

ORDER_S1, BOUNDS_S1 = [], []
for name, cols in GROUPS_S1:
    missing = [c for c in cols if c not in _s1.columns]
    assert not missing, f"features ausentes: {missing}"
    BOUNDS_S1.append((name, len(ORDER_S1), len(ORDER_S1) + len(cols)))
    ORDER_S1 += cols

corr_s1 = _s1[ORDER_S1].corr(method="pearson")  # pares completos
n_s1 = _s1[ORDER_S1].notna().astype(int).T @ _s1[ORDER_S1].notna().astype(int)

fig, ax = plt.subplots(figsize=(12.0, 11.5))
sns.heatmap(corr_s1, ax=ax, cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            square=True, xticklabels=True, yticklabels=True,
            cbar_kws={"label": "Pearson correlation", "shrink": .6})
ax.grid(False)  # o estilo paper liga a grade; no heatmap ela polui as células
ax.tick_params(axis="both", labelsize=5.5, length=0)
for _, start, _end in BOUNDS_S1[1:]:
    ax.axhline(start, color="black", lw=0.9)
    ax.axvline(start, color="black", lw=0.9)
# Rótulo de cada grupo acima da matriz (variável de entrada).
for name, start, end in BOUNDS_S1:
    ax.text((start + end) / 2, -0.8, name, ha="left", va="bottom",
            rotation=45, fontsize=8, fontweight="bold",
            rotation_mode="anchor")
for lab in (*ax.get_xticklabels(), *ax.get_yticklabels()):
    if lab.get_text() in ("qbal", "SRI12"):
        lab.set_color("#ad5528"); lab.set_fontweight("bold")
fig.tight_layout()
# 600 dpi: figura densa (77×77) do material suplementar, impressa em página inteira
save_figure(fig, FIG_DIR / "article_ct_total_h3_feature_correlation_S1.png",
            dpi=600)
corr_s1.to_csv(SHAP_DIR / "article_ct_total_h3_feature_correlation_S1.csv")

# Resumo para o texto suplementar.
print(pd.DataFrame(BOUNDS_S1, columns=["grupo", "ini", "fim"])
      .assign(n=lambda d: d["fim"] - d["ini"]).to_string(index=False))
print(f"Features candidatas (sem Vol): {len(ORDER_S1) - 1}")
print(f"Meses na janela: {len(_s1)}; n por par: min {int(n_s1.values.min())}, "
      f"Vol×tws {int(n_s1.loc['Vol', 'tws'])}, "
      f"Vol×qelev_Lag12 {int(n_s1.loc['Vol', 'qelev_Lag12'])}")
print("\nTop-12 |r| com Vol:")
print(corr_s1["Vol"].drop("Vol").abs().sort_values(ascending=False)
      .head(12).round(3).to_string())
plt.show()

## § 4. Previsões — por base de reconstrução e melhor modelo

Duas visões: primeiro os quatro feature sets em cada base (A/B/C/D na
persistência; A\*/B\*/C\*/D\* na climatologia), depois o **melhor modelo**
isolado (série + dispersão).

In [ ]:
# Figura: os 4 feature sets em cada base — (a) persistência A/B/C/D;
# (b) climatologia A*/B*/C*/D*. Observado + baselines (persistência-3, clim).
cores_fs = {"A": "#d97742", "B": "#7b5aa6", "C": "#2a9d8f", "D": "#c9a227"}
_te = preds_all[preds_all["split"] == "test"]
_dates = test_best["target_date"]
_obs = test_best["y_true"].to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(11.5, 8.2), sharex=True, sharey=True)
for ax, base, titulo in [
    (axes[0], "pers",
     "(a) A/B/C/D — persistence reconstruction: V̂ = Vol(t) + Δ̂"),
    (axes[1], "clim",
     "(b) A*/B*/C*/D* — climatology reconstruction: V̂ = Clim[m] + Δ̂"),
]:
    ax.plot(_dates, _obs, color="#1f3a5f", lw=2.0, label="observed", zorder=7)
    for fs_key, fs in EXP_SHORT.items():
        g = (_te[(_te["base"] == base) & (_te["experiment"] == fs_key)]
             .sort_values("target_date"))
        ax.plot(g["target_date"], g["y_pred"], color=cores_fs[fs], lw=1.4,
                label=(fs if base == "pers" else fs + "*"), zorder=5)
    ax.plot(_dates, persist3, color="#999999", lw=1.0, linestyle="--",
            label="persistence-3", zorder=3)
    ax.plot(_dates, clim_saz, color="#4a7a4a", lw=1.1, linestyle="-.",
            label="climatology", zorder=3)
    ax.set_title(titulo, loc="left", fontweight="bold")
    ax.set_ylabel("CT_TOTAL volume (hm³)")
    ax.legend(loc="upper left", frameon=False, fontsize=8, ncol=3)
    ax.grid(True, alpha=0.25)
axes[1].set_xlabel("target date")
fig.suptitle("CT_TOTAL — XGBoost H=3: experiments by reconstruction base "
             "(test, fixed split)", fontweight="bold", x=0.02, ha="left", y=1.0)
fig.tight_layout()
save_figure(fig, FIG_DIR / "article_ct_total_h3_experiments_by_base.png", dpi=300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6),
                         gridspec_kw={"width_ratios": [2, 1]})

ax = axes[0]
ax.plot(test_best["target_date"], test_best["y_true"], color="#1f3a5f",
        lw=2.0, label="observed")
ax.plot(test_best["target_date"], test_best["y_pred"], color="#d97742",
        lw=1.6, label=f"XGBoost ({BEST_LABEL})")
ax.plot(test_best["target_date"], persist3, color="#999999",
        lw=1.0, linestyle="--", label="persistence-3")
ax.plot(test_best["target_date"], clim_saz, color="#4a7a4a",
        lw=1.0, linestyle="-.", label="seasonal climatology")
ax.set_title("(a) CT_TOTAL — best model on test (H=3, fixed split)",
             loc="left", fontweight="bold")
ax.set_ylabel("CT_TOTAL volume (hm³)")
ax.set_xlabel("target date")
ax.legend(loc="upper left", frameon=False, fontsize=8)
ax.grid(True, alpha=0.25)

ax = axes[1]
lims = [test_best[["y_true", "y_pred"]].min().min(),
        test_best[["y_true", "y_pred"]].max().max()]
ax.scatter(test_best["y_true"], test_best["y_pred"], s=22, color="#d97742",
           edgecolor="black", linewidth=0.3, alpha=0.85)
ax.plot(lims, lims, color="0.35", lw=1.0, linestyle=":")
ax.set_title("(b) Observed vs predicted", loc="left", fontweight="bold")
ax.set_xlabel("observed (hm³)")
ax.set_ylabel("predicted (hm³)")
ax.grid(True, alpha=0.25)

fig.suptitle(f"CT_TOTAL — XGBoost H=3 (best: {BEST_LABEL})",
             fontweight="bold", x=0.02, ha="left", y=1.03)
fig.tight_layout()
save_figure(fig, FIG_DIR / "article_ct_total_h3_timeseries_scatter.png",
            dpi=300)
plt.show()

## § 5. Interpretabilidade — SHAP: o passo intermediário para os feature sets da LSTM

Reajustamos o experimento vencedor (mesmos hiperparâmetros/splits) e
computamos SHAP (`TreeExplainer`) no conjunto de teste. Leitura **não
causal** — mesma cautela do `docs/methodological_rules.md` §3.6: o SHAP
decompõe a função ajustada pelo modelo, não estabelece causalidade; sinais
contraintuitivos podem refletir a feature atuando como *proxy* de regime
sazonal/hidroclimático mais amplo.

In [ ]:
# Reajuste do vencedor para SHAP, respeitando a base de reconstrução:
# 'pers' -> delta sobre Vol; 'clim' -> level sobre a série de anomalias.
_shap_mode = "delta" if BEST_BASE == "pers" else "level"
_shap_vols = vol_total if BEST_BASE == "pers" else vol_anom
xgb_best = XGBoostModel(target_col="Vol", add_seasonality=True,
                        target_mode=_shap_mode)
fitted_best = fit_single_experiment_for_shap(
    xgb=xgb_best,
    datasets_by_basin=datasets_total,
    volumes=_shap_vols,
    basin="CT_TOTAL",
    horizon=CONFIG["H"],
    min_date=CONFIG["MIN_DATE"],
    max_date=CONFIG["MAX_DATE"],
    split_mode=CONFIG["SPLIT_MODE"],
    train_end=CONFIG["TRAIN_END"],
    val_end=CONFIG["VAL_END"],
    experiment=BEST_EXP,
    feature_cols=feature_sets_ct_total()[BEST_EXP],
    core_required=CONFIG["CORE_REQUIRED_MIN"],
    target_lags=CONFIG["TARGET_LAGS"],
    split_on=CONFIG["SPLIT_ON"],
    scale_mode="zscore",
    scale_raw_vars=CONFIG["SCALE_RAW_VARS"],
    scale_include_lags=True,
)
shap_values, X_eval, _ = compute_tree_shap(fitted_best, split_name="test")
shap_importance = shap_importance_table(shap_values, X_eval)
shap_group = shap_group_importance_table(shap_values, X_eval)

print(f"Top-10 features (SHAP, |mean|): "
      f"{shap_importance.head(10)['feature'].tolist()}")
shap_importance.head(15).round(4)

In [ ]:
plot_shap_importance_bar(
    shap_importance, top_n=15,
    title=f"CT_TOTAL — {BEST_LABEL}, H=3 — SHAP feature importance",
    output_path=FIG_DIR / "shap" / "article_ct_total_h3_shap_bar.png",
)
plot_shap_group_bar(
    shap_group,
    title=f"CT_TOTAL — {BEST_LABEL}, H=3 — SHAP by group",
    output_path=FIG_DIR / "shap" / "article_ct_total_h3_shap_group.png",
)
save_shap_beeswarm(
    shap_values, max_display=15,
    title=f"CT_TOTAL — {BEST_LABEL}, H=3 — SHAP beeswarm",
    output_path=FIG_DIR / "shap" / "article_ct_total_h3_shap_beeswarm.png",
)
plt.show()

### 5.1. Leitura — hierarquia de importância e direção para a LSTM

No vencedor `D`, o ranking SHAP: **`pr` (≈ 26 %), o calendário (`month_cos`
≈ 22 %) e `qbal` (≈ 17 %) dominam**, com `Vol_Lag1`, `is_dry` e `eto` em
seguida; o `SRI12` fica **fora do top‑8** (redundante com `q`/`qbal`). Por
grupo: calendário ≈ 31 %, precipitação ≈ 30 %, **operação (`qbal`) ≈ 17 %**,
armazenamento local ≈ 9 %, vazão natural ≈ 6 % e evapotranspiração ≈ 6 %.

**O contraste correlação × SHAP é o ponto para o artigo.** `SRI12`, que
correlaciona forte com o *nível* (ρ ≈ 0,75, §3.3), tem importância SHAP baixa;
já `qbal`, **quase não correlacionado com o nível** (ρ ≈ 0,03), é o **3º
driver** — porque informa a *variação* de volume (alvo `delta`), não o nível.
A importância de feature **não** se lê pela correlação linear.

Direção para a LSTM
([`lstm_article.ipynb`](../LSTM/lstm_article.ipynb) §4): o
sinal está concentrado em **precipitação + calendário + balanço operacional
agregado**. Nota importante e honesta: **não** é que toda variável operacional
prejudique — o balanço *agregado* `qbal` é um dos três maiores drivers; é o
conjunto operacional **completo** B (túneis e vazões locais) que introduz
ruído e perde para C e D.

### 5.2. SHAP no regime de seca

**O que define “seca” nesta análise (definição PADRONIZADA).** Seca = mês do
teste cujo **volume observado no alvo** (`y_true` em `target_date`) fica **abaixo do
limiar sazonal Q20** — o **percentil 20 do volume do CT_TOTAL por mês-calendário**,
estimado **apenas no treino** (sem vazamento). É **exatamente a mesma definição**
usada nas métricas de detecção (POD/FAR/HSS/AUC), nas figuras do artigo (Figs 3–5)
e no braço LSTM (`DROUGHT_Q = 0.20`). Substitui o critério antigo (quartil global
Q25 do `Vol` de entrada padronizado), agora **unificado em todo o trabalho**.

> Nota conceitual: aqui a condição é sobre o **resultado observado** (o alvo foi uma
> seca sazonal?), não sobre o **estado de entrada**. As linhas de seca são alinhadas
> **por posição** às linhas de teste do SHAP (mesma ordem de `X_eval`).

**Como foi feito (3 passos).**

1. **SHAP do vencedor.** Reajusta o experimento vencedor (`D`, base persistência ⇒
   modo `delta`) e computa o SHAP (TreeExplainer) sobre a matriz de **teste** `X_eval`;
   os valores vivem no espaço da **variação** ΔVol.
2. **Importância condicional à seca.** Aplica a máscara `y_true < Q20_sazonal[mês]` e
   recalcula a **média de |SHAP| por feature** nas amostras de seca, comparando ao
   teste completo (deslocamento de _rank_).
3. **Drivers de queda × de sustentação.** Decompõe o SHAP médio **com sinal** na seca:
   negativo empurra a ΔVol para baixo (aprofunda a queda); positivo sustenta o volume.

Leitura **interpretativa/associativa, não causal** (§5).

In [ ]:
# Máscara de seca — PADRONIZADA: Q20 sazonal sobre o volume OBSERVADO no alvo
# (percentil 20 por mês-calendário, estimado só no treino), igual às Figs 3-5
# e ao braço LSTM. Alinhada por posição às linhas de teste do SHAP.
sv_total = (shap_values.values if hasattr(shap_values, "values")
            else np.asarray(shap_values))
feat_names = list(X_eval.columns)

DROUGHT_Q = 0.20
_pdf = fitted_best.pred_df
_thr20 = (_pdf[_pdf["split"] == "train"]
          .assign(m=lambda d: d["target_date"].dt.month)
          .groupby("m")["y_true"].quantile(DROUGHT_Q))
_pdf_test = _pdf[_pdf["split"] == "test"].reset_index(drop=True)
assert len(_pdf_test) == len(X_eval), "SHAP-test desalinhado de pred_df-test"
mask_drought = (_pdf_test["y_true"].to_numpy()
                < _pdf_test["target_date"].dt.month.map(_thr20).to_numpy())
n_dr = int(mask_drought.sum())
print(f"Seca (obs < Q20 sazonal do treino): {n_dr}/{len(mask_drought)} "
      f"({100 * n_dr / len(mask_drought):.0f}% do teste)")

shap_dr_abs = np.abs(sv_total[mask_drought]).mean(axis=0)
imp_drought = (pd.DataFrame({"feature": feat_names,
                            "shap_drought": shap_dr_abs})
               .sort_values("shap_drought", ascending=False)
               .reset_index(drop=True))
imp_cmp = (shap_importance[["feature", "mean_abs_shap"]]
           .rename(columns={"mean_abs_shap": "shap_all"})
           .merge(imp_drought, on="feature", how="left")
           .assign(rank_all=lambda d: d["shap_all"].rank(ascending=False).astype(int),
                   rank_drought=lambda d: d["shap_drought"].rank(ascending=False).astype(int))
           .sort_values("shap_drought", ascending=False)
           .head(10).reset_index(drop=True))
imp_cmp["delta_rank"] = imp_cmp["rank_all"] - imp_cmp["rank_drought"]
print(imp_cmp[["feature", "shap_drought", "shap_all",
               "rank_drought", "rank_all", "delta_rank"]].round(4).to_string())

fig, ax = plt.subplots(figsize=(7.6, 4.2))
x, w = np.arange(len(imp_cmp)), 0.38
ax.barh(x - w / 2, imp_cmp["shap_all"], w, label="Full test",
        color="#0072B2", alpha=.85)
ax.barh(x + w / 2, imp_cmp["shap_drought"], w,
        label="Drought (observed ≤ monthly Q20)", color="#D55E00", alpha=.85)
ax.set_yticks(x); ax.set_yticklabels(imp_cmp["feature"]); ax.invert_yaxis()
ax.set_xlabel("mean |SHAP|")
ax.set_title(f"CT_TOTAL — {BEST_LABEL} — SHAP: full test "
             "vs. drought period", loc="left", fontweight="bold")
ax.legend(frameon=False); fig.tight_layout()
save_figure(fig, FIG_DIR / "shap" / "article_ct_total_h3_shap_drought_importance.png",
            dpi=300)
imp_drought.to_csv(SHAP_DIR / "article_ct_total_h3_shap_drought_importance.csv",
                   index=False)
plt.show()

In [ ]:
# SHAP médio COM SINAL nas amostras de seca (modo delta):
# negativo empurra ΔVol para baixo (queda); positivo sustenta o volume.
shap_dr_signed = sv_total[mask_drought].mean(axis=0)
fall_drivers = (pd.DataFrame({"feature": feat_names,
                             "mean_shap": shap_dr_signed})
                .sort_values("mean_shap").reset_index(drop=True))
top_neg = fall_drivers.head(8)
top_pos = fall_drivers.tail(8).iloc[::-1]
plot_df = (pd.concat([top_neg, top_pos]).drop_duplicates("feature")
           .sort_values("mean_shap"))

fig, ax = plt.subplots(figsize=(7.6, 4.6))
colors = ["#D55E00" if v < 0 else "#0072B2" for v in plot_df["mean_shap"]]
ax.barh(range(len(plot_df)), plot_df["mean_shap"], color=colors, alpha=.85)
ax.set_yticks(range(len(plot_df))); ax.set_yticklabels(plot_df["feature"])
ax.axvline(0, color="black", lw=.8, ls="--")
ax.set_xlabel(f"mean signed SHAP (winner {BEST_LABEL})")
ax.set_title(f"CT_TOTAL — decline drivers during drought "
             "(observed ≤ monthly Q20)", loc="left", fontweight="bold")
ax.annotate("← pushes decline", xy=(.02, .26), xycoords="axes fraction",
            color="#D55E00", fontsize=8)
ax.annotate("sustains volume →", xy=(.60, .26), xycoords="axes fraction",
            color="#0072B2", fontsize=8)
fig.tight_layout()
save_figure(fig, FIG_DIR / "shap" / "article_ct_total_h3_shap_fall_drivers.png",
            dpi=300)
fall_drivers.to_csv(SHAP_DIR / "article_ct_total_h3_shap_fall_drivers.csv",
                    index=False)
plt.show()

## § 6. Exportação de resultados

In [ ]:
ablation_ct_total.to_csv(
    PRED_DIR / "article_ct_total_h3_ablation.csv", index=False)
perf_ct_total.to_csv(
    PRED_DIR / "article_ct_total_h3_performance.csv", index=False)
comparacao.to_csv(
    PRED_DIR / "article_ct_total_h3_comparison.csv", index=False)
dm_ct_total.to_csv(
    PRED_DIR / "article_ct_total_h3_diebold_mariano.csv", index=False)
preds_best.to_csv(
    PRED_DIR / "article_ct_total_h3_preds.csv", index=False)
shap_importance.to_csv(
    SHAP_DIR / "article_ct_total_h3_shap_importance.csv", index=False)
shap_group.to_csv(
    SHAP_DIR / "article_ct_total_h3_shap_group.csv", index=False)

print("Exportação concluída:")
print(f"  predições/ablation -> {PRED_DIR}  (article_ct_total_h3_*.csv)")
print(f"  SHAP               -> {SHAP_DIR}  (article_ct_total_h3_*.csv)")
print(f"  figuras            -> {FIG_DIR}  (article_ct_total_h3_*.png)")

---
*Fim do notebook.* O repositório também contém um arquivo
[`docs/methodological_rules.md`](../../../docs/methodological_rules.md) com as diretrizes metodológicas
inegociáveis usadas nesta pesquisa.